# 1. Cleaning Data PetFinder

Tujuan notebook ini:
1. Mengambil data **anjing** saja dari dataset PetFinder.my (versi ringkas TensorFlow).
2. Membuang deskripsi yang kosong atau bukan bahasa Inggris (karena sentimen memakai VADER).
3. Menandai profil yang berisi lebih dari satu anjing.
4. Mengubah kolom teks (ukuran, bulu, kesehatan) menjadi angka untuk clustering dan regresi.
5. Membuat kolom **energi** dan **ramah anak** dari kata kunci di deskripsi.
6. Menyimpan hasilnya ke `data/dogs_clean.csv`.

In [1]:
import re
import pandas as pd

pd.set_option("display.max_colwidth", 80)

## Membaca dataset

In [2]:
df = pd.read_csv("../data/petfinder-mini.csv")
print("Ukuran data:", df.shape)
df.head()

Ukuran data: (11537, 15)


,Type,Age,Breed1,Gender,Color1,Color2,MaturitySize,FurLength,Vaccinated,Sterilized,Health,Fee,Description,PhotoAmt,AdoptionSpeed
0,Cat,3,Tabby,Male,Black,White,Small,Short,No,No,Healthy,100,Nibble is a 3+ month old ball of cuteness. He is energetic and playful. I re...,1,2
1,Cat,1,Domestic Medium Hair,Male,Black,Brown,Medium,Medium,Not Sure,Not Sure,Healthy,0,I just found it alone yesterday near my apartment. It was shaking so I had t...,2,0
2,Dog,1,Mixed Breed,Male,Brown,White,Medium,Medium,Yes,No,Healthy,0,Their pregnant mother was dumped by her irresponsible owner at the roadside ...,7,3
3,Dog,4,Mixed Breed,Female,Black,Brown,Medium,Short,Yes,No,Healthy,150,"Good guard dog, very alert, active, obedience waiting for her good master, p...",8,2
4,Dog,1,Mixed Breed,Male,Black,No Color,Medium,Short,No,No,Healthy,0,This handsome yet cute boy is up for adoption. He is the most playful pal we...,3,2


## 1. Ambil anjing saja

In [3]:
print(df["Type"].value_counts())

dogs = df[df["Type"] == "Dog"].copy()
print("\nJumlah anjing:", len(dogs))

Type
Dog    6601
Cat    4936
Name: count, dtype: int64

Jumlah anjing: 6601


## 2. Cek data kosong dan buang deskripsi kosong

In [4]:
print(dogs.isna().sum())

dogs["Description"] = dogs["Description"].fillna("").str.strip()
kosong = dogs["Description"] == ""
print("\nDeskripsi kosong:", kosong.sum())
dogs = dogs[~kosong]

Type             0
Age              0
Breed1           0
Gender           0
Color1           0
Color2           0
MaturitySize     0
FurLength        0
Vaccinated       0
Sterilized       0
Health           0
Fee              0
Description      2
PhotoAmt         0
AdoptionSpeed    0
dtype: int64

Deskripsi kosong: 2


## 3. Buang deskripsi yang bukan bahasa Inggris

Sebagian deskripsi berbahasa Mandarin. Cirinya, banyak huruf di luar huruf Latin (non-ASCII). Deskripsi dengan huruf ASCII kurang dari 90% dibuang, karena VADER hanya bisa membaca bahasa Inggris.

In [5]:
def porsi_ascii(teks):
    return sum(ch.isascii() for ch in teks) / len(teks)

dogs["porsi_ascii"] = dogs["Description"].apply(porsi_ascii)
bukan_inggris = dogs["porsi_ascii"] < 0.9
print("Deskripsi bukan bahasa Inggris:", bukan_inggris.sum())
print(dogs.loc[bukan_inggris, "Description"].head(3).tolist())

dogs = dogs[~bukan_inggris].drop(columns="porsi_ascii")
print("\nSisa anjing:", len(dogs))

Deskripsi bukan bahasa Inggris: 71
['Lost Dog Found (Bandar Menjalara, Kepong／Taman Bukit Maluri or Desa Park City) 迷途犬被尋獲 (甲洞孟加拉惹／武吉馬魯里／Desa Park City) Area lost puppy found 小狗被發現地點：Bandar Sri Menjalara, Kepong, KL, Malaysia 大馬吉隆坡甲洞孟加拉惹 How the lost dog found 狗兒如何被發現: The lost puppy entered my staying place itself. 迷途小狗自行進我居住地。 Condition 情況: 雖好像幾天沒進食。看起來健康。頸上系有紅帶。目前甲洞孟加拉惹住宅區保安處看顧。Few days without food. Looks healthy. Neck tied with red belt. Currently taken care by security guard of housing area of Bandar Menjalara.', "Dear all dog lovers~ I'm helping my friend to share the post. She rescued the puppies & they waiting for a Home~ Condition : - Must bring them to spay when over 6 months old. - Must bring them to vet vaccine . - Must promise not to abandoned & abuse. - If not suitable please contact rescuer & don't simply give to other people without let rescuer know. - Must allow to visit half a year or rescuer decide. - Must keep contact with rescuer. - Truly care & love them. Contact

## 4. Tandai profil berisi beberapa anjing

Versi ringkas ini tidak punya kolom jumlah hewan per profil. Profil yang menyebut *puppies*, *litter*, atau *siblings* ditandai dengan kolom `grup = True`. Data ini tidak dibuang, tapi bisa dikecualikan saat melatih regresi.

In [6]:
pola_grup = r"\b(?:puppies|litter|siblings)\b"
dogs["grup"] = dogs["Description"].str.lower().str.contains(pola_grup, regex=True)
print(dogs["grup"].value_counts())

grup
False    5799
True      729
Name: count, dtype: int64


## 5. Ubah kolom teks menjadi angka

In [7]:
dogs["ukuran_angka"] = dogs["MaturitySize"].map({"Small": 1, "Medium": 2, "Large": 3})
dogs["bulu_angka"] = dogs["FurLength"].map({"Short": 1, "Medium": 2, "Long": 3})
dogs["kesehatan_angka"] = dogs["Health"].map({"Healthy": 0, "Minor Injury": 1, "Serious Injury": 2})

dogs[["MaturitySize", "ukuran_angka", "FurLength", "bulu_angka", "Health", "kesehatan_angka"]].drop_duplicates().head(10)

,MaturitySize,ukuran_angka,FurLength,bulu_angka,Health,kesehatan_angka
2,Medium,2,Medium,2,Healthy,0
3,Medium,2,Short,1,Healthy,0
11,Medium,2,Long,3,Healthy,0
23,Large,3,Short,1,Healthy,0
39,Small,1,Short,1,Minor Injury,1
42,Large,3,Medium,2,Healthy,0
52,Small,1,Short,1,Healthy,0
59,Small,1,Long,3,Healthy,0
111,Large,3,Medium,2,Minor Injury,1
206,Medium,2,Short,1,Minor Injury,1


## 6. Energi dan ramah anak dari kata kunci deskripsi

- **Energi tinggi (3)** kalau ada kata seperti *active, energetic, playful, hyper, lively*.
- **Energi rendah (1)** kalau ada kata seperti *calm, quiet, lazy, gentle, relaxed*.
- **Energi sedang (2)** kalau tidak ada keduanya atau ada keduanya.
- **Ramah anak = Ya** kalau ada frasa seperti *good with kids*. Selain itu **Belum tahu**, karena deskripsi jarang menyebutnya.

In [8]:
pola_tinggi = r"\b(?:active|energetic|playful|hyper|lively)\b"
pola_rendah = r"\b(?:calm|quiet|lazy|gentle|relaxed)\b"
pola_anak = r"good with (?:kids|children)|(?:kid|child)[- ]friendly|loves? (?:kids|children)"

teks = dogs["Description"].str.lower()
ada_tinggi = teks.str.contains(pola_tinggi, regex=True)
ada_rendah = teks.str.contains(pola_rendah, regex=True)

def tentukan_energi(tinggi, rendah):
    if tinggi and not rendah:
        return 3
    if rendah and not tinggi:
        return 1
    return 2

dogs["energi"] = [tentukan_energi(t, r) for t, r in zip(ada_tinggi, ada_rendah)]
dogs["energi_label"] = dogs["energi"].map({1: "Rendah", 2: "Sedang", 3: "Tinggi"})
dogs["ramah_anak"] = teks.str.contains(pola_anak, regex=True).map({True: "Ya", False: "Belum tahu"})

print(dogs["energi_label"].value_counts())
print()
print(dogs["ramah_anak"].value_counts())

energi_label
Sedang    4628
Tinggi    1521
Rendah     379
Name: count, dtype: int64

ramah_anak
Belum tahu    6461
Ya              67
Name: count, dtype: int64


## 7. Ringkasan data bersih

In [9]:
dogs = dogs.reset_index(drop=True)
dogs.insert(0, "anjing_id", dogs.index + 1)

print("Jumlah anjing bersih:", len(dogs))
print("\nUmur (bulan):")
print(dogs["Age"].describe())
print("\nKecepatan adopsi:")
print(dogs["AdoptionSpeed"].value_counts().sort_index())

Jumlah anjing bersih: 6528

Umur (bulan):
count    6528.000000
mean       14.366575
std        22.520371
min         0.000000
25%         2.000000
50%         4.000000
75%        16.000000
max       255.000000
Name: Age, dtype: float64

Kecepatan adopsi:
AdoptionSpeed
0     145
1    1202
2    1747
3    1600
4    1834
Name: count, dtype: int64


## 8. Simpan hasil

In [10]:
kolom = [
    "anjing_id", "Age", "Breed1", "Gender", "Color1", "Color2",
    "MaturitySize", "ukuran_angka", "FurLength", "bulu_angka",
    "Vaccinated", "Sterilized", "Health", "kesehatan_angka",
    "Fee", "PhotoAmt", "Description", "grup",
    "energi", "energi_label", "ramah_anak", "AdoptionSpeed",
]
dogs[kolom].to_csv("../data/dogs_clean.csv", index=False)
print("Tersimpan:", len(dogs), "baris,", len(kolom), "kolom")

Tersimpan: 6528 baris, 22 kolom
